In [ ]:
!pip install openai langchain faiss-cpu tiktoken

In [ ]:
from langchain_openai import ChatOpenAI, OpenAIEmbeddings
from langchain.vectorstores import FAISS
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.docstore.document import Document
from langchain.agents import create_react_agent, AgentExecutor, Tool
from langchain.prompts import ChatPromptTemplate
from langchain.memory import ConversationBufferMemory
from langchain.document_loaders import PyPDFLoader

In [ ]:
# Embeddings + LLM
embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
llm = ChatOpenAI(model="gpt-4", temperature=0)

# Sample KB
# 2️⃣ Load PDF
loader = PyPDFLoader("HR_Policy_Manual_2023.pdf")
docs = loader.load()
print(f"Loaded {len(docs)} pages")

splitter = RecursiveCharacterTextSplitter(chunk_size=200, chunk_overlap=20)
chunks = splitter.split_documents(docs)

vectorstore = FAISS.from_documents(chunks, embeddings)
retriever = vectorstore.as_retriever()

In [ ]:
memory = ConversationBufferMemory(
    memory_key="chat_history",
    return_messages=True
)

In [ ]:
def retrieve_tool(query: str):
    docs = retriever.invoke(query)
    return "\n".join([d.page_content for d in docs[:3]])  # limit results


tools = [
    Tool(
        name="KnowledgeRetriever",
        func=lambda q: "\n".join([d.page_content for d in retrieve_tool(q)]),
        description="Useful for retrieving domain knowledge from FAISS."
    )
]

In [ ]:
prompt = ChatPromptTemplate.from_messages([
    ("system", "You are an agent that answers questions using tools and memory."),
    ("human", "{input}"),
    ("ai", "{agent_scratchpad}"),
    ("system", "Available tools:\n{tools}\nTool names: {tool_names}")
])
# Create a React agent
agent = create_react_agent(
    llm=llm,
    tools=tools,
    prompt=prompt
)

# Wrap in AgentExecutor
agent_executor = AgentExecutor.from_agent_and_tools(
    agent=agent,
    tools=tools,
    memory=memory,
    verbose=True,
    handle_parsing_errors=True,
    max_interation=3
)

In [ ]:
response1 = agent_executor.invoke({"input": "What is  Casual Leave in India?"})
print(response1["output"])

response2 = agent_executor.invoke({"input": "How many Casual Leave can apply in India??"})
print(response2["output"])

response3 = agent_executor.invoke({"input": "What is HRMS system in India?"})
print(response3["output"])